
# Fine‑Tuning GPT‑2 for Machine Translation


### 1. Install and import dependencies

In [1]:
!pip -q install transformers datasets sentencepiece sacrebleu accelerate

In [2]:

import math, os, random
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    get_linear_schedule_with_warmup,
)

import matplotlib.pyplot as plt
from sacrebleu import corpus_bleu
from pathlib import Path

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Mixed precision (AMP) will be enabled.')


PyTorch: 2.10.0+cpu
CUDA available: False


### 2. Configuration & reproducibility

In [3]:

MODEL_NAME = 'gpt2'               # # smllest GPT-2 version
SRC_LANG = 'English'
TGT_LANG = 'French'
PROMPT_TPL = 'translate English to French: '
SEP_TOKEN = '<|sep|>'              # separates source and target
LANG_TOKENS = ['<|en|>', '<|fr|>'] # optional language tags

MAX_LENGTH = 160                   # total sequence length (prompt + src + sep + tgt)
BATCH_SIZE = 8
NUM_EPOCHS = 2                     # keep small for class time
LEARNING_RATE = 5e-5               # conservative for GPT‑2 FT
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_ACCUM_STEPS = 2
MAX_NEW_TOKENS = 80                # generation length for target side
TEMPERATURE = 1.0                  # parameters for sampling
TOP_P = 0.95
NUM_BEAMS = 4                      # you can switch to beam search; set top_p=None if using beams

# Subsampling for classroom runtime
SUBSET_TRAIN = 6000
SUBSET_VALID = 800

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('gpt2_en2de_runs'); OUT_DIR.mkdir(parents=True, exist_ok=True)


### 3. Load a parallel corpus for transalation (OPUS Books: English→French)

In this example we will be using the [*OPUS books*](https://huggingface.co/datasets/Helsinki-NLP/opus_books). This is a collection of copyright free books multilingually aligned for 16 languages. 

In [4]:
# We load the english-french portion of the OPUS Books dataset
raw_dset = load_dataset("opus_books", "en-fr")
print(raw_dset)

VALIDATION_RATIO = 0.2
split = raw_dset["train"].train_test_split(test_size=VALIDATION_RATIO, seed=SEED)
train_raw = split["train"]
valid_raw = split["test"]

# Apply subsampling if configured
if SUBSET_TRAIN is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(SUBSET_TRAIN, len(train_raw))))
if SUBSET_VALID is not None:
    valid_raw = valid_raw.shuffle(seed=SEED).select(range(min(SUBSET_VALID, len(valid_raw))))

print("Train size:", len(train_raw), "| Valid size:", len(valid_raw))


DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
Train size: 6000 | Valid size: 800


### 4. Tokenization & preprocessing
We use **GPT-2** tokenizer to tokenize the dataset. GPT-2 has **no PAD token** by default, so we set `pad_token = eos_token`. We also add a separator token `<|sep|>` and optional language tags to clearly split source/target.


In [6]:

# Load tokenizer and add special tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
added = tokenizer.add_special_tokens({'additional_special_tokens': [SEP_TOKEN] + LANG_TOKENS})

# Ensure we have a pad token; use EOS as PAD for GPT‑2
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Added tokens:', added)
print('PAD token id:', tokenizer.pad_token_id, 'EOS id:', tokenizer.eos_token_id)


Added tokens: 3
PAD token id: 50256 EOS id: 50256



####  Preprocessing function
For each example, construct an input string:

```
"translate English to French: " + <|en|> + SRC + <|sep|> + <|fr|> + TGT
```

Then create labels that are `-100` for all tokens up to the end of the prompt (so the loss only trains on the target side).


In [ ]:
EN, FR = LANG_TOKENS

# Build the input text for a single example, given the source and target sentences following the template:
# "translate English to French: <en> {src} <|sep|> <fr> {tgt}"
# Returns the full text and the prompt 
def build_example(sample):
    src = sample['translation']['en']
    tgt = sample['translation']['fr']
    prompt = PROMPT_TPL + EN + ' ' + src + ' ' + SEP_TOKEN + ' ' + FR + ' '
    text = prompt + tgt
    return text, prompt


def preprocess_batch(batch):
    texts = []
    prompts = []
    # Build the full text and prompt for each example in the batch
    for ex in batch['translation']:
        t, p = build_example({'translation': ex})
        texts.append(t)
        prompts.append(p)

    # Tokenize the full text (prompt + src + sep + tgt) 
    enc = tokenizer(texts, max_length=MAX_LENGTH, truncation=True, padding=False)

    # FOR LABELS, WE WANT ONLY TO COMPUTE LOSS FOR THE TARGET TOKENS. HOW CAN WE IGNORE PROMPT, SOURCE AND SPECIAL TOKENS?
    # ADD CODE HERE




    return enc

train_tok = train_raw.map(preprocess_batch, batched=True, remove_columns=train_raw.column_names)
valid_tok = valid_raw.map(preprocess_batch, batched=True, remove_columns=valid_raw.column_names)

print(train_tok)
print(valid_tok)


Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})


### 5. Data Loader
We pad to the longest sequence in each batch, both inputs and labels. In the labels, pad tokens are converted to **-100** so the loss ignores them.

In [ ]:

def collate_fn(features):
    batch = tokenizer.pad(
        {k: [f[k] for f in features] for k in ['input_ids', 'attention_mask']},
        padding=True,
        return_tensors='pt'
    )

    # Pad the labels to the same length as input_ids, but use the label_pad_token_id (-100) so they are ignored in the loss
    labels_batch = {"input_ids": [f["labels"] for f in features]}
    padded_labels = tokenizer.pad(labels_batch, padding=True, return_tensors="pt")["input_ids"]
    # Replace padding token id with -100 to ignore in loss computation
    padded_labels = padded_labels.masked_fill(padded_labels == tokenizer.pad_token_id, -100)
    batch["labels"] = padded_labels
    return batch

train_loader = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_tok, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_loader), len(valid_loader)


(750, 100)

### 6. Model, optimizer, and scheduler

We will be using [`GPT2LMHeadModel`](https://huggingface.co/docs/transformers/en/model_doc/openai-gpt#transformers.OpenAIGPTLMHeadModel), for text generation. 

The specific checkpoint of the model is specified by the parameter `MODEL_NAME`. By default, we will use 'gpt2', the smallest available GPT model with 124M parameters. You can also try larger versions 'gpt2-medium', 'gpt2-large' or 'gpt2-xl' (see https://huggingface.co/openai-community?search_models=gpt)

In [9]:
# We load the pretrained GPT-2 model for language generation
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
# Resize the model's token embeddings to account for the new special tokens we added to the tokenizer.
model.resize_token_embeddings(len(tokenizer))
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
t_total_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * t_total_steps)

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=t_total_steps)

print(f'Total steps: {t_total_steps} | Warmup steps: {num_warmup_steps}')


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Total steps: 750 | Warmup steps: 45


### 7. Inference with the pre-trained model

We do inference for the first 10 elements in the validation set to check the performanca of the pre-trained model before fine-tuning.

In [12]:
model.eval()

# First 10 validation examples from the raw parallel corpus
samples = valid_raw.select(range(10))

sources = [ex["translation"]["en"] for ex in samples]
targets = [ex["translation"]["fr"] for ex in samples]
prompts = [f"{PROMPT_TPL}{EN} {src} {SEP_TOKEN} {FR} " for src in sources]

# Tokenize prompts as a batch
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

with torch.no_grad():
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS if NUM_BEAMS > 1 else 1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=False)

for i, (src, tgt, out_text) in enumerate(zip(sources, targets, decoded), 1):
    # Extract generated target text after separator
    if SEP_TOKEN in out_text:
        pred = out_text.split(SEP_TOKEN, 1)[-1]
    else:
        pred = out_text

    # Remove optional FR tag and common special tokens
    pred = pred.replace(FR, "").replace(tokenizer.eos_token, "").replace(tokenizer.pad_token, "").strip()

    print(f"Example {i}")
    print("Source   :", src)
    print("Target   :", tgt)
    print("Predicted:", pred)
    print("-" * 80)

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Example 1
Source   : Therese then saw what a terrible shock her aunt had received.
Target   : Thérèse put voir quel terrible coup avait reçu sa tante.
Predicted: . 𐐐���𐐐�������������������������������������������������������������������
--------------------------------------------------------------------------------
Example 2
Source   : "Ah!" cried Neb, "if my master was here, he would know what to do!"
Target   : -- Ah! s'écria Nab, s'il était là, mon maître, il saurait bien vous en faire!»
Predicted: "Ah!" cried Neb, "if my master was here, he would know what to do!" 𐐐���𐐐𐐐�𐐐�𐐐��𐐐��𐐐��𐐐��𐐐��𐐐��
--------------------------------------------------------------------------------
Example 3
Source   : As for our neglect, our isolation in the depths of this cell, I was afraid to guess at how long it might last.
Target   : Quant à notre abandon, notre isolement au fond de cette cellule, je n'osais estimer ce qu'il pourrait durer.
Predicted: and 𐐐��𐐐��������������������������������������������

### 8. Training loop


In [ ]:
train_loss_hist, valid_loss_hist, bleu_hist = [], [], []
best_bleu = -1.0

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    running = 0.0
    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f'Epoch {epoch} [train]')
    
    optimizer.zero_grad(set_to_none=True)
    for step, batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        out = model(**batch)
        # Scales the loss before backpropagation so that gradient accumulation matches the magnitude of a larger effective batch
        loss = out.loss / GRAD_ACCUM_STEPS
        loss.backward()
        
        # Only update weights and step scheduler every GRAD_ACCUM_STEPS to simulate larger batch size        
        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

       # For logging, we accumulate the loss scaled by GRAD_ACCUM_STEPS to reflect the effective batch size.   
        running += loss.item() * GRAD_ACCUM_STEPS
        avg_loss = running / step
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    train_loss_hist.append(running/len(train_loader))

    # Validation
    model.eval()
    val_running = 0.0
    preds, refs = [], []
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [valid]'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            out = model(**batch)
            val_running += out.loss.item()
            input_ids = batch['input_ids']
            
            # For evaluation, extract the prompt + source from the text and pass only this to the model
            prompt_texts = []
            for ids in input_ids:
                text = tokenizer.decode(ids.tolist(), skip_special_tokens=False)
                # Keep only the prompt/source side and prepare the target prefix
                if SEP_TOKEN in text:
                    prompt = text.split(SEP_TOKEN, 1)[0] + SEP_TOKEN + ' ' + FR + ' '
                else:
                    prompt = text
                prompt_texts.append(prompt)

            enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, truncation=True).to(DEVICE)

            # Generate translation using either beam search with NUM_BEAMS or random sampling
            # Check different options for sampling during generation in https://huggingface.co/docs/transformers/en/main_classes/text_generation
            gen = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=(NUM_BEAMS==1),
                    temperature=TEMPERATURE if NUM_BEAMS==1 else None,
                    top_p=TOP_P if NUM_BEAMS==1 else None,
                    num_beams=NUM_BEAMS if NUM_BEAMS>1 else 1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Remove special tokens from the prediction for evaluation
            pred_texts = tokenizer.batch_decode(gen, skip_special_tokens=False)
            for pred_text in pred_texts:
                if SEP_TOKEN in pred_text:
                    pred_tgt = pred_text.split(SEP_TOKEN, 1)[-1]
                else:
                    pred_tgt = pred_text

                pred_tgt = pred_tgt.replace(FR, "").replace(tokenizer.eos_token, "").replace(tokenizer.pad_token, "").strip()
                preds.append(pred_tgt)
            labels = batch['labels'].clone()
            # Convert back ids -100 with the PAD token for evaluation
            labels[labels==-100] = tokenizer.pad_token_id
            dec_refs = tokenizer.batch_decode(labels, skip_special_tokens=True)
            refs.extend(map(str.strip, dec_refs))

    bleu = corpus_bleu(preds, [refs])
    mean_val_loss = val_running/len(valid_loader)
    valid_loss_hist.append(mean_val_loss)
    bleu_hist.append(bleu.score)
    print(f'Epoch {epoch}: train_loss={train_loss_hist[-1]:.4f} | valid_loss={mean_val_loss:.4f} | BLEU={bleu.score:.2f}')

    if bleu.score > best_bleu:
        best_bleu = bleu.score
        save_dir = 'gpt2-en2de-best'
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print('Saved new best model to', save_dir)

# Save histories
np.save(OUT_DIR/'train_loss.npy', np.array(train_loss_hist))
np.save(OUT_DIR/'valid_loss.npy', np.array(valid_loss_hist))
np.save(OUT_DIR/'bleu.npy', np.array(bleu_hist))


Epoch 1 [train]:   0%|          | 0/750 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 9. Plot metrics across epochs
We visualize **training/validation loss** and **BLEU** score.

In [ ]:

if 'train_loss_hist' not in globals():
    train_loss_hist = np.load(OUT_DIR/'train_loss.npy').tolist()
    valid_loss_hist = np.load(OUT_DIR/'valid_loss.npy').tolist()
    bleu_hist = np.load(OUT_DIR/'bleu.npy').tolist()

epochs = list(range(1, len(train_loss_hist)+1))

plt.figure(figsize=(6,4))
plt.plot(epochs, train_loss_hist, label='Training loss')
plt.plot(epochs, valid_loss_hist, label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss curves'); plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/'loss_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs, bleu_hist, label='Validation BLEU')
plt.xlabel('Epoch'); plt.ylabel('BLEU'); plt.title('Validation BLEU'); plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/'bleu_curve.png', dpi=150)
plt.show()

print('Saved figures to', OUT_DIR.resolve())


### 10. Inference with the fine-tuned model

We do inference for the first 10 elements in the validation set to check the performanca of the fine-tuned model.

In [ ]:
model.eval()

# First 10 validation examples from the raw parallel corpus
samples = valid_raw.select(range(10))

sources = [ex["translation"]["en"] for ex in samples]
targets = [ex["translation"]["fr"] for ex in samples]
prompts = [f"{PROMPT_TPL}{EN} {src} {SEP_TOKEN} {FR} " for src in sources]

# Tokenize prompts as a batch
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

with torch.no_grad():
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS if NUM_BEAMS > 1 else 1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=False)

for i, (src, tgt, out_text) in enumerate(zip(sources, targets, decoded), 1):
    # Extract generated target text after separator
    if SEP_TOKEN in out_text:
        pred = out_text.split(SEP_TOKEN, 1)[-1]
    else:
        pred = out_text

    # Remove optional FR tag and common special tokens
    pred = pred.replace(FR, "").replace(tokenizer.eos_token, "").replace(tokenizer.pad_token, "").strip()

    print(f"Example {i}")
    print("Source   :", src)
    print("Target   :", tgt)
    print("Predicted:", pred)
    print("-" * 80)